# NeuroBridge-S4 Graph Learning — Phase 4: Interpretable Graph Feature Extraction

**Phase 3** constructed biological adaptation graphs — one per pseudo-crew participant.  
**Phase 4** makes those graphs measurable and comparable.

A graph becomes scientifically useful when we can measure its structure.  
Phase 4 extracts interpretable graph features that answer HRP-relevant questions:

- Which biological domains are most activated?
- Which domains are most central in the graph?
- Is activation concentrated in one system or distributed across several?
- Which biological subgraph appears most involved?

---

## Scientific guardrails

> **These features are research interpretation features.**  
> They are **not diagnostic**.  
> They do **not prove causality**.  
> They do **not predict astronaut health**.  
> They prepare this project for graph embeddings and novelty detection in Phase 5.

This notebook uses processed proxy data from NeuroBridge-S4.  
It does **not** use actual Artemis II astronaut data.

---
## 1. Setup

In [1]:
import sys
from pathlib import Path

# Ensure src/ is on the path
repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for notebook/CI
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import networkx as nx

from neurobridge_graph.graph_features import (
    extract_all_graph_level_features,
    extract_all_node_level_features,
    extract_all_edge_level_features,
)
from neurobridge_graph.subgraph_features import extract_all_subgraph_features

# Output directories
RESULTS = repo_root / 'results'
for d in ['tables', 'figures', 'reports']:
    (RESULTS / d).mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


---
## 2. Load Phase 3 graphs

GraphML files are loaded from `results/graphs/`.  
If they are not present, Phase 3 graphs are rebuilt from processed data.

In [2]:
graphs_dir = RESULTS / 'graphs'
graphml_files = sorted(graphs_dir.glob('subject_graph_*.graphml'))

graphs = {}

if graphml_files:
    for gf in graphml_files:
        G = nx.read_graphml(gf)
        # Restore float types for numeric attributes (GraphML stores as strings)
        for node, attrs in G.nodes(data=True):
            for k in ('activation', 'domain_score'):
                if k in attrs:
                    try:
                        attrs[k] = float(attrs[k])
                    except (ValueError, TypeError):
                        pass
        for u, v, attrs in G.edges(data=True):
            for k in ('weight',):
                if k in attrs:
                    try:
                        attrs[k] = float(attrs[k])
                    except (ValueError, TypeError):
                        pass
        sid = G.graph.get('subject_id', gf.stem.replace('subject_graph_', ''))
        graphs[sid] = G
    print(f'Loaded {len(graphs)} GraphML files from {graphs_dir}')
else:
    print('GraphML files not found — rebuilding from processed data ...')
    from neurobridge_graph.data_loader import load_domain_scores, load_baci_scores
    from neurobridge_graph.graph_builder import build_all_subject_graphs
    domain_scores = load_domain_scores()
    baci_df = load_baci_scores()
    graphs = build_all_subject_graphs(domain_scores, baci_df)
    print(f'Built {len(graphs)} graphs from processed data.')

print(f'\nSubject IDs: {list(graphs.keys())}')
for sid, G in graphs.items():
    print(f'  {sid}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

Loaded 4 GraphML files from /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/graphs

Subject IDs: ['Crew 100987', 'Crew 94465', 'Crew 97774', 'Crew 99813']
  Crew 100987: 6 nodes, 4 edges
  Crew 94465: 6 nodes, 4 edges
  Crew 97774: 6 nodes, 5 edges
  Crew 99813: 6 nodes, 4 edges


---
## 3. Graph-level features

Graph-level features summarise each participant as one biological adaptation graph.  
They allow comparison across pseudo-crew members without looking at individual nodes.

In [3]:
graph_features_df = extract_all_graph_level_features(graphs)
save_path = RESULTS / 'tables' / 'graph_level_features.csv'
graph_features_df.to_csv(save_path, index=False)
print(f'Saved: {save_path}')
print(f'Shape: {graph_features_df.shape}')
graph_features_df

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/tables/graph_level_features.csv
Shape: (4, 18)


,subject_id,n_nodes,n_edges,graph_density,mean_node_activation,median_node_activation,max_node_activation,total_node_activation,n_active_domains,active_domain_fraction,mean_edge_weight,max_edge_weight,conceptual_edge_count,coactivation_edge_count,top_domain,top_domain_activation,baci_score,baci_category
0,Crew 100987,6,4,0.26667,0.84097,0.68918,1.88191,5.04580,1,0.16667,1.00000,1.00000,4,0,Cardiovascular regulation,1.88191,28.6,mild coherence
1,Crew 94465,6,4,0.26667,0.83948,0.83517,1.32253,5.03688,1,0.16667,1.00000,1.00000,4,0,Body composition / physical status,1.32253,22.1,low coherence
2,Crew 97774,6,5,0.33333,1.05170,0.94380,2.10729,6.31018,2,0.33333,1.19746,1.98728,4,1,Hematologic / oxygen-carrying,2.10729,41.5,mild coherence
3,Crew 99813,6,4,0.26667,0.69729,0.83698,1.15790,4.18376,2,0.33333,1.00000,1.00000,4,0,Metabolic regulation,1.15790,31.5,mild coherence


---
## 4. Node-level features

Node-level features show which biological domains are activated and central for each participant.

> **Note:** Activation and centrality are different.  
> A domain can be highly activated but not central (no edges connecting it to others),  
> or central but not highly activated (structurally connected but biologically quiet).

In [4]:
node_features_df = extract_all_node_level_features(graphs)
save_path = RESULTS / 'tables' / 'node_level_features.csv'
node_features_df.to_csv(save_path, index=False)
print(f'Saved: {save_path}')
print(f'Shape: {node_features_df.shape}')
node_features_df.head(12)

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/tables/node_level_features.csv
Shape: (24, 11)


,subject_id,domain,activation,activation_level,domain_score,degree,weighted_degree,degree_centrality,is_active,is_top_domain,interpretation
0,Crew 100987,Cardiovascular regulation,1.88191,high,1.88191,2,2.0,0.4,True,True,Cardiovascular regulation: high domain activat...
1,Crew 100987,Metabolic regulation,0.68918,low,0.68918,3,3.0,0.6,False,False,Metabolic regulation: reference-relative signa...
2,Crew 100987,Body composition / physical status,0.66846,low,0.66846,1,1.0,0.2,False,False,Body composition / physical status: reference-...
3,Crew 100987,Inflammation / immune-adjacent,0.48617,low,0.48617,1,1.0,0.2,False,False,Inflammation / immune-adjacent: reference-rela...
4,Crew 100987,Hematologic / oxygen-carrying,0.61494,low,0.61494,1,1.0,0.2,False,False,Hematologic / oxygen-carrying: reference-relat...
5,Crew 100987,Recovery-related markers,0.70515,low,0.70515,0,0.0,0.0,False,False,Recovery-related markers: reference-relative s...
6,Crew 94465,Cardiovascular regulation,0.94861,mild,0.94861,2,2.0,0.4,False,False,Cardiovascular regulation: mild reference-rela...
7,Crew 94465,Metabolic regulation,0.83517,mild,0.83517,3,3.0,0.6,False,False,Metabolic regulation: mild reference-relative ...
8,Crew 94465,Body composition / physical status,1.32253,moderate,1.32253,1,1.0,0.2,True,True,Body composition / physical status: moderate d...
9,Crew 94465,Inflammation / immune-adjacent,0.49645,low,0.49645,1,1.0,0.2,False,False,Inflammation / immune-adjacent: reference-rela...


---
## 5. Edge-level features

Edge-level features preserve edge type information.  
**Conceptual edges** and **co-activation edges** are not the same:  
- Conceptual edges reflect established biological relationships.  
- Co-activation edges reflect that two domains were both elevated in the same participant.

> Neither implies causal proof.

In [5]:
edge_features_df = extract_all_edge_level_features(graphs)
save_path = RESULTS / 'tables' / 'edge_level_features.csv'
edge_features_df.to_csv(save_path, index=False)
print(f'Saved: {save_path}')
print(f'Shape: {edge_features_df.shape}')
edge_features_df.head(10)

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/tables/edge_level_features.csv
Shape: (17, 8)


,subject_id,source,target,edge_type,weight,connects_active_domains,relationship,interpretation
0,Crew 100987,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.0,False,Cardiovascular and metabolic systems share reg...,Conceptual biological relationship; not causal...
1,Crew 100987,Cardiovascular regulation,Hematologic / oxygen-carrying,conceptual_biological_relationship,1.0,False,Oxygen-carrying capacity is tightly coupled to...,Conceptual biological relationship; not causal...
2,Crew 100987,Metabolic regulation,Body composition / physical status,conceptual_biological_relationship,1.0,False,Metabolic state and body composition are bidir...,Conceptual biological relationship; not causal...
3,Crew 100987,Metabolic regulation,Inflammation / immune-adjacent,conceptual_biological_relationship,1.0,False,Inflammatory signalling modulates metabolic pa...,Conceptual biological relationship; not causal...
4,Crew 94465,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.0,False,Cardiovascular and metabolic systems share reg...,Conceptual biological relationship; not causal...
5,Crew 94465,Cardiovascular regulation,Hematologic / oxygen-carrying,conceptual_biological_relationship,1.0,False,Oxygen-carrying capacity is tightly coupled to...,Conceptual biological relationship; not causal...
6,Crew 94465,Metabolic regulation,Body composition / physical status,conceptual_biological_relationship,1.0,False,Metabolic state and body composition are bidir...,Conceptual biological relationship; not causal...
7,Crew 94465,Metabolic regulation,Inflammation / immune-adjacent,conceptual_biological_relationship,1.0,False,Inflammatory signalling modulates metabolic pa...,Conceptual biological relationship; not causal...
8,Crew 97774,Cardiovascular regulation,Metabolic regulation,conceptual_biological_relationship,1.0,False,Cardiovascular and metabolic systems share reg...,Conceptual biological relationship; not causal...
9,Crew 97774,Cardiovascular regulation,Hematologic / oxygen-carrying,conceptual_biological_relationship,1.0,False,Oxygen-carrying capacity is tightly coupled to...,Conceptual biological relationship; not causal...


---
## 6. Subgraph-level features

Subgraph features ask whether activation is concentrated in biologically meaningful clusters.

Templates used:
- **cardiometabolic** — cardiovascular, metabolic, body composition
- **immune_metabolic** — inflammation, metabolic, recovery markers
- **hematologic_cardiovascular** — hematologic, cardiovascular, recovery markers
- **sleep_autonomic_recovery** — sleep/circadian, autonomic, recovery capacity
- **cognitive_emotional_recovery** — cognitive load, emotional regulation, recovery

Missing template nodes do not cause errors — available nodes are reported.

In [6]:
subgraph_features_df = extract_all_subgraph_features(graphs)
save_path = RESULTS / 'tables' / 'subgraph_features.csv'
subgraph_features_df.to_csv(save_path, index=False)
print(f'Saved: {save_path}')
print(f'Shape: {subgraph_features_df.shape}')
subgraph_features_df[[
    'subject_id', 'subgraph_name', 'n_available_nodes',
    'n_active_nodes', 'subgraph_activation_mean',
    'dominant_node'
]]

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/tables/subgraph_features.csv
Shape: (20, 11)


,subject_id,subgraph_name,n_available_nodes,n_active_nodes,subgraph_activation_mean,dominant_node
0,Crew 100987,cardiometabolic,3,1,1.07985,Cardiovascular regulation
1,Crew 100987,immune_metabolic,3,0,0.62683,Recovery-related markers
2,Crew 100987,hematologic_cardiovascular,3,1,1.06733,Cardiovascular regulation
3,Crew 100987,sleep_autonomic_recovery,0,0,NaN,n/a
4,Crew 100987,cognitive_emotional_recovery,1,0,0.70515,Recovery-related markers
5,Crew 94465,cardiometabolic,3,1,1.03544,Body composition / physical status
6,Crew 94465,immune_metabolic,3,0,0.71267,Metabolic regulation
7,Crew 94465,hematologic_cardiovascular,3,0,0.79424,Cardiovascular regulation
8,Crew 94465,sleep_autonomic_recovery,0,0,NaN,n/a
9,Crew 94465,cognitive_emotional_recovery,1,0,0.80638,Recovery-related markers


---
## 7. Visualisations

All figures are saved to `results/figures/`.

In [7]:
# ---- Helper: short participant labels ----
def short_label(sid: str) -> str:
    """Convert 'Crew 97774' or 'Crew_97774' to 'P-97774'."""
    import re
    num = re.sub(r'(?i)^crew[_ ]*', '', str(sid)).strip()
    return f'P-{num}' if num else sid

subject_labels = {sid: short_label(sid) for sid in graph_features_df['subject_id']}
print('Participant labels:', subject_labels)

Participant labels: {'Crew 100987': 'P-100987', 'Crew 94465': 'P-94465', 'Crew 97774': 'P-97774', 'Crew 99813': 'P-99813'}


In [8]:
# ---- Figure 1: Graph-level feature comparison bar chart ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels = [subject_labels.get(sid, sid) for sid in graph_features_df['subject_id']]
x = np.arange(len(labels))
bar_w = 0.6

# Panel 1: max node activation
ax = axes[0]
bars = ax.bar(x, graph_features_df['max_node_activation'], width=bar_w,
              color='#F5CBA7', edgecolor='#C0392B', linewidth=0.8)
ax.axhline(1.0, color='#888', linestyle='--', linewidth=0.8, label='Active threshold (1.0)')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Max domain activation', fontsize=9)
ax.set_title('Max Domain Activation per Participant', fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(0, max(graph_features_df['max_node_activation'].max() * 1.2, 1.5))
for bar, val in zip(bars, graph_features_df['max_node_activation']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8)

# Panel 2: number of active domains
ax = axes[1]
bars2 = ax.bar(x, graph_features_df['n_active_domains'], width=bar_w,
               color='#AED6F1', edgecolor='#2471A3', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Number of active domains', fontsize=9)
ax.set_title('Active Domains per Participant', fontsize=10)
ax.set_ylim(0, max(graph_features_df['n_active_domains'].max() + 1, 3))
for bar, val in zip(bars2, graph_features_df['n_active_domains']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(int(val)), ha='center', va='bottom', fontsize=9)

fig.suptitle(
    'Phase 4 — Graph-level feature comparison\n'
    '(Research interpretation only. Not diagnostic.)',
    fontsize=9, color='#555', y=1.01
)
plt.tight_layout()
out_path = RESULTS / 'figures' / 'phase4_graph_level_feature_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out_path}')

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/figures/phase4_graph_level_feature_comparison.png


In [9]:
# ---- Figure 2: Node activation heatmap ----
pivot_act = node_features_df.pivot_table(
    index='subject_id', columns='domain', values='activation', aggfunc='first'
)
row_labels = [subject_labels.get(sid, sid) for sid in pivot_act.index]

fig, ax = plt.subplots(figsize=(max(8, len(pivot_act.columns) * 1.2), max(3, len(pivot_act) * 0.8 + 1.5)))
im = ax.imshow(pivot_act.values, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xticks(np.arange(len(pivot_act.columns)))
ax.set_xticklabels(pivot_act.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(np.arange(len(pivot_act)))
ax.set_yticklabels(row_labels, fontsize=9)
ax.set_title('Node activation by domain and participant\n(Research interpretation only)', fontsize=10)
plt.colorbar(im, ax=ax, label='Activation (reference-relative)', shrink=0.8)
# Annotate cells
for r in range(len(pivot_act)):
    for c in range(len(pivot_act.columns)):
        val = pivot_act.values[r, c]
        if not np.isnan(val):
            ax.text(c, r, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if val > 1.2 else '#333')
plt.tight_layout()
out_path = RESULTS / 'figures' / 'phase4_node_activation_heatmap.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out_path}')

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/figures/phase4_node_activation_heatmap.png


In [10]:
# ---- Figure 3: Node centrality heatmap ----
pivot_cent = node_features_df.pivot_table(
    index='subject_id', columns='domain', values='degree_centrality', aggfunc='first'
)
row_labels_c = [subject_labels.get(sid, sid) for sid in pivot_cent.index]

fig, ax = plt.subplots(figsize=(max(8, len(pivot_cent.columns) * 1.2), max(3, len(pivot_cent) * 0.8 + 1.5)))
im = ax.imshow(pivot_cent.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(np.arange(len(pivot_cent.columns)))
ax.set_xticklabels(pivot_cent.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(np.arange(len(pivot_cent)))
ax.set_yticklabels(row_labels_c, fontsize=9)
ax.set_title('Node degree centrality by domain and participant\n(Research interpretation only)', fontsize=10)
plt.colorbar(im, ax=ax, label='Degree centrality', shrink=0.8)
for r in range(len(pivot_cent)):
    for c in range(len(pivot_cent.columns)):
        val = pivot_cent.values[r, c]
        if not np.isnan(val):
            ax.text(c, r, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if val > 0.6 else '#333')
plt.tight_layout()
out_path = RESULTS / 'figures' / 'phase4_node_centrality_heatmap.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out_path}')

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/figures/phase4_node_centrality_heatmap.png


In [11]:
# ---- Figure 4: Subgraph activation mean heatmap ----
valid_sub = subgraph_features_df[subgraph_features_df['n_available_nodes'] > 0].copy()

if len(valid_sub) > 0:
    pivot_sg = valid_sub.pivot_table(
        index='subject_id', columns='subgraph_name',
        values='subgraph_activation_mean', aggfunc='first'
    )
    row_labels_s = [subject_labels.get(sid, sid) for sid in pivot_sg.index]

    fig, ax = plt.subplots(figsize=(max(7, len(pivot_sg.columns) * 1.5), max(3, len(pivot_sg) * 0.8 + 1.5)))
    data_vals = pivot_sg.values.astype(float)
    # Replace NaN with 0 for display
    display_vals = np.where(np.isnan(data_vals), 0.0, data_vals)
    im = ax.imshow(display_vals, cmap='PuOr_r', aspect='auto', vmin=0)
    ax.set_xticks(np.arange(len(pivot_sg.columns)))
    ax.set_xticklabels(pivot_sg.columns, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(pivot_sg)))
    ax.set_yticklabels(row_labels_s, fontsize=9)
    ax.set_title('Subgraph activation mean by template and participant\n(Research interpretation only)', fontsize=10)
    plt.colorbar(im, ax=ax, label='Mean activation in subgraph', shrink=0.8)
    for r in range(len(pivot_sg)):
        for c in range(len(pivot_sg.columns)):
            v = data_vals[r, c]
            label = f'{v:.2f}' if not np.isnan(v) else 'n/a'
            ax.text(c, r, label, ha='center', va='center', fontsize=8,
                    color='white' if (not np.isnan(v) and v > 1.0) else '#333')
    plt.tight_layout()
    out_path = RESULTS / 'figures' / 'phase4_subgraph_activation_heatmap.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out_path}')
else:
    print('No valid subgraph features to plot.')

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/figures/phase4_subgraph_activation_heatmap.png


---
## 8. Interpretation report

In [12]:
import datetime

n_graphs = len(graphs)
gf = graph_features_df

# Participant with highest max activation
idx_max = gf['max_node_activation'].idxmax()
top_act_participant = gf.loc[idx_max, 'subject_id']
top_act_value = gf.loc[idx_max, 'max_node_activation']

# Participant with broadest activation (most active domains)
idx_broad = gf['n_active_domains'].idxmax()
broad_participant = gf.loc[idx_broad, 'subject_id']
broad_n_active = gf.loc[idx_broad, 'n_active_domains']

# Most frequently active domain
nf = node_features_df.copy()
nf['is_active'] = nf['activation'] >= 1.0
domain_active_counts = nf.groupby('domain')['is_active'].sum().sort_values(ascending=False)
most_freq_active_domain = domain_active_counts.index[0] if len(domain_active_counts) > 0 else 'n/a'

# Dominant subgraph per participant
sf = subgraph_features_df[subgraph_features_df['n_available_nodes'] > 0].copy()
dominant_subgraphs = sf.loc[
    sf.groupby('subject_id')['subgraph_activation_mean'].idxmax()
][['subject_id', 'subgraph_name', 'subgraph_activation_mean']]

lines = [
    "NeuroBridge-S4 Graph Learning — Phase 4 Interpretation Report",
    "=" * 60,
    f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
    "",
    f"Graphs analysed: {n_graphs}",
    f"Participants: {', '.join(gf['subject_id'].tolist())}",
    "",
    "--- Activation summary ---",
    f"Highest max activation: {top_act_participant} ({top_act_value:.3f})",
    f"Broadest activation: {broad_participant} ({int(broad_n_active)} active domains)",
    f"Most frequently active domain: {most_freq_active_domain}",
    "",
    "--- Dominant subgraph per participant ---",
]
for _, row in dominant_subgraphs.iterrows():
    lines.append(
        f"  {row['subject_id']}: {row['subgraph_name']} "
        f"(mean activation {row['subgraph_activation_mean']:.3f})"
    )

lines += [
    "",
    "--- Interpretation guardrail ---",
    "These features are research interpretation outputs.",
    "They are not diagnostic. They do not prove causality.",
    "They do not predict astronaut health.",
    "",
    "--- Preparing for Phase 5 ---",
    "Phase 4 feature tables will serve as input to Phase 5 graph embeddings.",
    "Key tables:",
    "  results/tables/graph_level_features.csv",
    "  results/tables/node_level_features.csv",
    "  results/tables/edge_level_features.csv",
    "  results/tables/subgraph_features.csv",
    "In Phase 5, these will be used to build graph embeddings, similarity maps,",
    "and reference-relative novelty scores.",
]

report_text = "\n".join(lines)
report_path = RESULTS / 'reports' / 'phase4_graph_feature_report.txt'
report_path.write_text(report_text, encoding='utf-8')
print(f'Saved: {report_path}')
print()
print(report_text)

Saved: /Users/rm/Desktop/NeuroBridge-S4-Graph-Learning/results/reports/phase4_graph_feature_report.txt

NeuroBridge-S4 Graph Learning — Phase 4 Interpretation Report
Generated: 2026-06-06 09:20

Graphs analysed: 4
Participants: Crew 100987, Crew 94465, Crew 97774, Crew 99813

--- Activation summary ---
Highest max activation: Crew 97774 (2.107)
Broadest activation: Crew 97774 (2 active domains)
Most frequently active domain: Body composition / physical status

--- Dominant subgraph per participant ---
  Crew 100987: cardiometabolic (mean activation 1.080)
  Crew 94465: cardiometabolic (mean activation 1.035)
  Crew 97774: hematologic_cardiovascular (mean activation 1.178)
  Crew 99813: immune_metabolic (mean activation 1.031)

--- Interpretation guardrail ---
These features are research interpretation outputs.
They are not diagnostic. They do not prove causality.
They do not predict astronaut health.

--- Preparing for Phase 5 ---
Phase 4 feature tables will serve as input to Phase 5 g

---
## 9. Conclusion

**Phase 4 is complete.**

Biological adaptation graphs have been converted into four feature tables:

| Table | Content |
|-------|---------|
| `graph_level_features.csv` | One row per participant: density, activation stats, edge counts, BACI |
| `node_level_features.csv` | One row per domain per participant: activation, centrality, active flag |
| `edge_level_features.csv` | One row per edge per participant: type, weight, active flag |
| `subgraph_features.csv` | One row per template per participant: mean activation, dominant domain |

**Phase 5** will use these feature tables to build graph embeddings and similarity maps,  
allowing participants to be compared in graph space rather than raw biomedical value space.

> Research interpretation only. Not diagnosis or treatment guidance.